# CRUD Operations Helper

Reusable Python functions for Create, Read, Update, and Delete operations on every table in the `cps714.restaurant_manager` schema.

In [0]:
CATALOG = "cps714"
SCHEMA = "restaurant_manager"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

def _q(value):
    """Quote a Python value for safe interpolation into SQL."""
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    if isinstance(value, (int, float)):
        return str(value)
    # Escape single quotes by doubling them
    return "'" + str(value).replace("'", "''") + "'"

In [0]:
# ── Restaurant CRUD ─────────────────────────────────────────

def create_restaurant(name, address=None, phone=None):
    """Insert a new restaurant and return the generated restaurant_id."""
    spark.sql(f"""
        INSERT INTO restaurant (name, address, phone)
        VALUES ({_q(name)}, {_q(address)}, {_q(phone)})
    """)
    row = spark.sql("SELECT max(restaurant_id) AS id FROM restaurant").collect()[0]
    return row["id"]

def get_restaurant(restaurant_id=None):
    """Return all restaurants, or a single restaurant by ID."""
    if restaurant_id is not None:
        return spark.sql(f"SELECT * FROM restaurant WHERE restaurant_id = {restaurant_id}")
    return spark.sql("SELECT * FROM restaurant")

def update_restaurant(restaurant_id, **fields):
    """Update one or more columns of a restaurant. Pass column=value kwargs."""
    if not fields:
        raise ValueError("No fields provided to update")
    set_clause = ", ".join(f"{col} = {_q(val)}" for col, val in fields.items())
    spark.sql(f"UPDATE restaurant SET {set_clause} WHERE restaurant_id = {restaurant_id}")

def delete_restaurant(restaurant_id):
    """Delete a restaurant by ID."""
    spark.sql(f"DELETE FROM restaurant WHERE restaurant_id = {restaurant_id}")

In [0]:
# ── Restaurant Table CRUD ───────────────────────────────────

def create_restaurant_table(restaurant_id, table_number, capacity=4, status="available"):
    """Insert a new dining table for a restaurant."""
    spark.sql(f"""
        INSERT INTO restaurant_table (restaurant_id, table_number, capacity, status)
        VALUES ({restaurant_id}, {table_number}, {capacity}, {_q(status)})
    """)
    row = spark.sql("SELECT max(table_id) AS id FROM restaurant_table").collect()[0]
    return row["id"]

def get_restaurant_table(table_id=None, restaurant_id=None):
    """Return dining tables, optionally filtered by table_id or restaurant_id."""
    conditions = []
    if table_id is not None:
        conditions.append(f"table_id = {table_id}")
    if restaurant_id is not None:
        conditions.append(f"restaurant_id = {restaurant_id}")
    where = " WHERE " + " AND ".join(conditions) if conditions else ""
    return spark.sql(f"SELECT * FROM restaurant_table{where}")

def update_restaurant_table(table_id, **fields):
    """Update one or more columns of a dining table."""
    if not fields:
        raise ValueError("No fields provided to update")
    set_clause = ", ".join(f"{col} = {_q(val)}" for col, val in fields.items())
    spark.sql(f"UPDATE restaurant_table SET {set_clause} WHERE table_id = {table_id}")

def delete_restaurant_table(table_id):
    """Delete a dining table by ID."""
    spark.sql(f"DELETE FROM restaurant_table WHERE table_id = {table_id}")

In [0]:
# ── Seat CRUD ───────────────────────────────────────────────

def create_seat(table_id, seat_number):
    """Insert a new seat at a table."""
    spark.sql(f"""
        INSERT INTO seat (table_id, seat_number)
        VALUES ({table_id}, {seat_number})
    """)
    row = spark.sql("SELECT max(seat_id) AS id FROM seat").collect()[0]
    return row["id"]

def get_seat(seat_id=None, table_id=None):
    """Return seats, optionally filtered by seat_id or table_id."""
    conditions = []
    if seat_id is not None:
        conditions.append(f"seat_id = {seat_id}")
    if table_id is not None:
        conditions.append(f"table_id = {table_id}")
    where = " WHERE " + " AND ".join(conditions) if conditions else ""
    return spark.sql(f"SELECT * FROM seat{where}")

def update_seat(seat_id, **fields):
    """Update one or more columns of a seat."""
    if not fields:
        raise ValueError("No fields provided to update")
    set_clause = ", ".join(f"{col} = {_q(val)}" for col, val in fields.items())
    spark.sql(f"UPDATE seat SET {set_clause} WHERE seat_id = {seat_id}")

def delete_seat(seat_id):
    """Delete a seat by ID."""
    spark.sql(f"DELETE FROM seat WHERE seat_id = {seat_id}")

In [0]:
# ── Menu Item CRUD ──────────────────────────────────────────

def create_menu_item(restaurant_id, name, price, description=None, category=None, available=True):
    """Insert a new menu item."""
    spark.sql(f"""
        INSERT INTO menu_item (restaurant_id, name, description, price, category, available)
        VALUES ({restaurant_id}, {_q(name)}, {_q(description)}, {price}, {_q(category)}, {_q(available)})
    """)
    row = spark.sql("SELECT max(menu_item_id) AS id FROM menu_item").collect()[0]
    return row["id"]

def get_menu_item(menu_item_id=None, restaurant_id=None):
    """Return menu items, optionally filtered by menu_item_id or restaurant_id."""
    conditions = []
    if menu_item_id is not None:
        conditions.append(f"menu_item_id = {menu_item_id}")
    if restaurant_id is not None:
        conditions.append(f"restaurant_id = {restaurant_id}")
    where = " WHERE " + " AND ".join(conditions) if conditions else ""
    return spark.sql(f"SELECT * FROM menu_item{where}")

def update_menu_item(menu_item_id, **fields):
    """Update one or more columns of a menu item."""
    if not fields:
        raise ValueError("No fields provided to update")
    set_clause = ", ".join(f"{col} = {_q(val)}" for col, val in fields.items())
    spark.sql(f"UPDATE menu_item SET {set_clause} WHERE menu_item_id = {menu_item_id}")

def delete_menu_item(menu_item_id):
    """Delete a menu item by ID."""
    spark.sql(f"DELETE FROM menu_item WHERE menu_item_id = {menu_item_id}")

In [0]:
# ── Seat Order Item CRUD ────────────────────────────────────

def create_seat_order_item(seat_id, menu_item_id, quantity=1, status="pending"):
    """Insert a new seat order item."""
    spark.sql(f"""
        INSERT INTO seat_order_item (seat_id, menu_item_id, quantity, status)
        VALUES ({seat_id}, {menu_item_id}, {quantity}, {_q(status)})
    """)
    row = spark.sql("SELECT max(seat_order_id) AS id FROM seat_order_item").collect()[0]
    return row["id"]

def get_seat_order_item(seat_order_id=None, seat_id=None):
    """Return seat order items, optionally filtered by seat_order_id or seat_id."""
    conditions = []
    if seat_order_id is not None:
        conditions.append(f"seat_order_id = {seat_order_id}")
    if seat_id is not None:
        conditions.append(f"seat_id = {seat_id}")
    where = " WHERE " + " AND ".join(conditions) if conditions else ""
    return spark.sql(f"SELECT * FROM seat_order_item{where}")

def update_seat_order_item(seat_order_id, **fields):
    """Update one or more columns of a seat order item."""
    if not fields:
        raise ValueError("No fields provided to update")
    set_clause = ", ".join(f"{col} = {_q(val)}" for col, val in fields.items())
    spark.sql(f"UPDATE seat_order_item SET {set_clause} WHERE seat_order_id = {seat_order_id}")

def delete_seat_order_item(seat_order_id):
    """Delete a seat order item by ID."""
    spark.sql(f"DELETE FROM seat_order_item WHERE seat_order_id = {seat_order_id}")

In [0]:
# ── Employee CRUD ───────────────────────────────────────────

def create_employee(restaurant_id, first_name, last_name, role="server", phone=None, hire_date=None, status="active"):
    """Insert a new employee."""
    spark.sql(f"""
        INSERT INTO employee (restaurant_id, first_name, last_name, role, phone, hire_date, status)
        VALUES ({restaurant_id}, {_q(first_name)}, {_q(last_name)}, {_q(role)}, {_q(phone)}, {_q(hire_date)}, {_q(status)})
    """)
    row = spark.sql("SELECT max(employee_id) AS id FROM employee").collect()[0]
    return row["id"]

def get_employee(employee_id=None, restaurant_id=None):
    """Return employees, optionally filtered by employee_id or restaurant_id."""
    conditions = []
    if employee_id is not None:
        conditions.append(f"employee_id = {employee_id}")
    if restaurant_id is not None:
        conditions.append(f"restaurant_id = {restaurant_id}")
    where = " WHERE " + " AND ".join(conditions) if conditions else ""
    return spark.sql(f"SELECT * FROM employee{where}")

def update_employee(employee_id, **fields):
    """Update one or more columns of an employee."""
    if not fields:
        raise ValueError("No fields provided to update")
    set_clause = ", ".join(f"{col} = {_q(val)}" for col, val in fields.items())
    spark.sql(f"UPDATE employee SET {set_clause} WHERE employee_id = {employee_id}")

def delete_employee(employee_id):
    """Delete an employee by ID."""
    spark.sql(f"DELETE FROM employee WHERE employee_id = {employee_id}")